In [ ]:
import pandas as pd
import numpy as np

# Load Excel file
jester = pd.read_excel('jester-data-1.xls', header=None)

print(f"Shape: {jester.shape}")
print(jester.head())

Shape: (24983, 101)
   0      1      2      3      4     5     6     7     8      9    ...    91   \
0   74  -7.82   8.79  -9.66  -8.16 -7.52 -8.50 -9.85  4.17  -8.98  ...   2.82   
1  100   4.08  -0.29   6.36   4.37 -2.38 -9.66 -0.73 -5.34   8.88  ...   2.82   
2   49  99.00  99.00  99.00  99.00  9.03  9.27  9.03  9.27  99.00  ...  99.00   
3   48  99.00   8.35  99.00  99.00  1.80  8.16 -2.82  6.21  99.00  ...  99.00   
4   91   8.50   4.61  -4.17  -5.39  1.36  1.60  7.04  4.61  -0.44  ...   5.19   

     92     93     94     95     96     97     98     99     100  
0  99.00  99.00  99.00  99.00  99.00  -5.63  99.00  99.00  99.00  
1  -4.95  -0.29   7.86  -0.19  -2.14   3.06   0.34  -4.32   1.07  
2  99.00  99.00   9.08  99.00  99.00  99.00  99.00  99.00  99.00  
3  99.00  99.00   0.53  99.00  99.00  99.00  99.00  99.00  99.00  
4   5.58   4.27   5.19   5.73   1.55   3.11   6.55   1.80   1.60  

[5 rows x 101 columns]


In [ ]:
# Convert matrix to long format
rows = []
for user_idx, row in jester.iterrows():
    for joke_idx in range(1, 101):  # columns 1-100 are ratings
        rating = row[joke_idx]
        if rating != 99.00:  # 99 means not rated
            rows.append({
                'user_id': user_idx,
                'item_id': joke_idx,
                'rating': rating
            })

jester_df = pd.DataFrame(rows)
print(f"Total ratings: {jester_df.shape[0]}")
print(f"Unique users: {jester_df['user_id'].nunique()}")
print(f"Unique jokes: {jester_df['item_id'].nunique()}")
print(f"Rating range: {jester_df['rating'].min():.2f} to {jester_df['rating'].max():.2f}")

# Rescale from [-10, 10] to [1, 5]
jester_df['rating'] = (jester_df['rating'] + 10) / 20 * 4 + 1
print(f"Rescaled range: {jester_df['rating'].min():.2f} to {jester_df['rating'].max():.2f}")

Total ratings: 1810455
Unique users: 24983
Unique jokes: 100
Rating range: -9.95 to 10.00
Rescaled range: 1.01 to 5.00


In [ ]:
def laplace_mechanism(matrix, epsilon, mean_rating):
    sensitivity = 4.0
    noise = np.random.laplace(0, sensitivity/epsilon, matrix.shape)
    noisy_matrix = matrix + noise
    noisy_matrix = np.clip(noisy_matrix, 1, 5)
    return noisy_matrix

In [ ]:
import numpy as np

def piecewise_mechanism(x, epsilon):
    # Step 1: normalise x from [1,5] to [-1,1]
    x_norm = 2 * (x - 1) / (5 - 1) - 1
    # C = (e^(ε/2) + 1) / (e^(ε/2) - 1)
    c= (np.exp(epsilon/2)+1)/(np.exp(epsilon/2)-1)
    # p = (e^ε - e^(ε/2)) / (2 * e^(ε/2) + 2)
    p= (np.exp(epsilon)-np.exp(epsilon/2))/(2*np.exp(epsilon/2)+2)
    # l(x) = (C+1)/2 * x_norm - (C-1)/2
    # r(x) = l(x) + C - 1
    l= (c+1)/2*x_norm-(c-1)/2
    r=l+c-1
    # threshold = e^(ε/2) / (e^(ε/2) + 1)
    threshold = np.exp(epsilon/2)/(np.exp(epsilon/2)+1)
    if np.random.uniform(0, 1) < threshold:
    # heads — sample from centre zone
      output = np.random.uniform(l, r)
    else:
    # tails — sample from tails
    # left tail runs from -C to l
    # right tail runs from r to +C
    # pick one randomly
      output = np.random.uniform(-c, l) if np.random.uniform(0,1) < 0.5 else np.random.uniform(r, c)
    x_out = (output + 1) * 2 + 1
    x_out = np.clip(x_out, 1, 5)
    return x_out

In [ ]:
def apply_piecewise(matrix, epsilon, mean_rating):
    noisy_matrix = matrix.copy()
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if matrix[i, j] != mean_rating:
                noisy_matrix[i, j] = piecewise_mechanism(matrix[i, j], epsilon)
    return noisy_matrix

In [ ]:
def ndcg_at_k(test, predicted_matrix, train_matrix, k=10, mean_rating=0):
    ndcg_scores = []

    # group test ratings by user
    test_grouped = test.groupby('user_id')

    for user_id, user_test in test_grouped:
        if user_id not in train_matrix.index:
            continue

        user_idx = train_matrix.index.get_loc(user_id)

        # get predicted scores for all items in test set for this user
        actual_ratings = []
        predicted_scores = []

        for _, row in user_test.iterrows():
            item_id = row['item_id']
            actual = row['rating']

            if item_id in train_matrix.columns:
                item_idx = train_matrix.columns.get_loc(item_id)
                predicted = predicted_matrix[user_idx, item_idx]
            else:
                predicted = mean_rating

            actual_ratings.append(actual)
            predicted_scores.append(predicted)

        if len(actual_ratings) < 2:
            continue

        # sort by predicted score to get top K
        paired = sorted(zip(predicted_scores, actual_ratings),
                       reverse=True)[:k]

        # compute DCG
        dcg = sum(rating / np.log2(rank + 2)
                 for rank, (_, rating) in enumerate(paired))

        # compute ideal DCG
        ideal_paired = sorted(actual_ratings, reverse=True)[:k]
        idcg = sum(rating / np.log2(rank + 2)
                  for rank, rating in enumerate(ideal_paired))

        if idcg > 0:
            ndcg_scores.append(dcg / idcg)

    return np.mean(ndcg_scores) if ndcg_scores else 0.0

In [ ]:
def gaussian_mechanism(matrix, epsilon, delta, mean_rating):
    sensitivity = 4.0
    sigma = sensitivity * np.sqrt(2 * np.log(1.25 / delta)) / epsilon
    noise = np.random.normal(0, sigma, matrix.shape)
    noisy_matrix = matrix + noise
    noisy_matrix = np.clip(noisy_matrix, 1, 5)
    return noisy_matrix

In [ ]:
def bounded_laplace_mechanism(matrix, epsilon, mean_rating):
    sensitivity = 1.0  # normalised scale [0,1]
    b = sensitivity / epsilon
    noisy_matrix = matrix.copy()
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if matrix[i, j] != mean_rating:
                r = (matrix[i, j] - 1) / 4  # normalise to [0,1]
                while True:
                    noise = np.random.laplace(0, b)
                    r_noisy = r + noise
                    if 0 <= r_noisy <= 1:
                        break
                noisy_matrix[i, j] = r_noisy * 4 + 1  # denormalise
    return noisy_matrix

In [ ]:
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error

n_splits = 10
#datadf_50 = jester_df.sample(frac=0.5, random_state=42)
datadf_20 = jester_df.sample(frac=0.2, random_state=42)
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
for fold, (train_idx, test_idx) in enumerate(kf.split(datadf_20)):
    train = datadf_20.iloc[train_idx]
    test = datadf_20.iloc[test_idx]
all_results = []
epsilons = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
sparsity_results = []

for fold, (train_idx, test_idx) in enumerate(kf.split(datadf_20)):
    print(f"Fold {fold+1}/{n_splits}")

    # split data
    train = datadf_20.iloc[train_idx]
    test = datadf_20.iloc[test_idx]
    print(f"Train size: {len(train)}, Test size: {len(test)}")

    try:
        train_matrix = train.pivot(index='user_id',
                                   columns='item_id',
                                   values='rating')
        print(f"Train matrix shape: {train_matrix.shape}")
    except Exception as e:
        print(f"Error building pivot at fold {fold}: {e}")
        continue
    # build matrix
    #train_matrix = train.pivot(index='user_id',
    #                           columns='item_id',
    #                           values='rating')
    train_array = train_matrix.values.astype(float)
    mean_rating = np.nanmean(train_array)
    print(f"Mean rating: {mean_rating:.4f}, NaN count: {np.isnan(train_array).sum()}")
    train_array[np.isnan(train_array)] = mean_rating
    print(f"NaN after fill: {np.isnan(train_array).sum()}")

    test_users = test['user_id'].values
    test_items = test['item_id'].values
    test_ratings = test['rating'].values
    gaussian_with_delta = lambda matrix, eps, mean_rating: gaussian_mechanism(matrix, eps, 1e-5, mean_rating)
    for eps in epsilons:
      for mechanism_name, mechanism_fn in [
        ('clipped_laplace', laplace_mechanism),
        ('bounded_laplace', bounded_laplace_mechanism),
        ('gaussian', gaussian_with_delta),
        ('piecewise', apply_piecewise)
      ]:
        try:
            noisy = mechanism_fn(train_array, eps, mean_rating)
            noisy = np.nan_to_num(noisy, nan=mean_rating,
                                  posinf=5.0, neginf=1.0)

            svd = TruncatedSVD(n_components=20, random_state=42)
            reduced = svd.fit_transform(noisy)
            predicted_matrix = np.dot(reduced, svd.components_)
            predicted_matrix = np.nan_to_num(predicted_matrix,
                                             nan=mean_rating)

            predicted_ratings = []
            for user, item in zip(test_users, test_items):
                if user in train_matrix.index and item in train_matrix.columns:
                    u_idx = train_matrix.index.get_loc(user)
                    i_idx = train_matrix.columns.get_loc(item)
                    predicted_ratings.append(predicted_matrix[u_idx, i_idx])
                else:
                    predicted_ratings.append(mean_rating)

            predicted_ratings = np.array(predicted_ratings)
            predicted_ratings = np.nan_to_num(predicted_ratings,
                                              nan=mean_rating)
            rmse = np.sqrt(mean_squared_error(test_ratings,
                                              predicted_ratings))
            ndcg = ndcg_at_k(test, predicted_matrix, train_matrix, k=10,
                  mean_rating=mean_rating)

            all_results.append({
              'fold': fold,
              'epsilon': eps,
              'mechanism': mechanism_name,
              'rmse': rmse,
              'ndcg': ndcg
            })

            print(f"Fold {fold+1}, {mechanism_name}, ε={eps}: RMSE={rmse:.4f}, NDCG@10={ndcg:.4f}")

        except Exception as e:
            print(f"ERROR at fold {fold+1}, {mechanism_name}, ε={eps}: {e}")
            continue

# compute mean and std across folds
results_df = pd.DataFrame(all_results)
summary = results_df.groupby(['epsilon', 'mechanism']).agg(
    rmse_mean=('rmse', 'mean'),
    rmse_std=('rmse', 'std'),
    ndcg_mean=('ndcg', 'mean'),
    ndcg_std=('ndcg', 'std')
).reset_index()
print(summary)

Fold 1/10
Train size: 325881, Test size: 36210
Train matrix shape: (24981, 100)
Mean rating: 3.1774, NaN count: 2172219
NaN after fill: 0
Fold 1, clipped_laplace, ε=0.1: RMSE=1.3960, NDCG@10=0.9539
Fold 1, bounded_laplace, ε=0.1: RMSE=1.0614, NDCG@10=0.9548
Fold 1, gaussian, ε=0.1: RMSE=1.4022, NDCG@10=0.9540
Fold 1, piecewise, ε=0.1: RMSE=1.0705, NDCG@10=0.9563
Fold 1, clipped_laplace, ε=0.5: RMSE=1.3489, NDCG@10=0.9543
Fold 1, bounded_laplace, ε=0.5: RMSE=1.0597, NDCG@10=0.9549
Fold 1, gaussian, ε=0.5: RMSE=1.3887, NDCG@10=0.9533
Fold 1, piecewise, ε=0.5: RMSE=1.0609, NDCG@10=0.9565
Fold 1, clipped_laplace, ε=1.0: RMSE=1.2960, NDCG@10=0.9550
Fold 1, bounded_laplace, ε=1.0: RMSE=1.0537, NDCG@10=0.9558
Fold 1, gaussian, ε=1.0: RMSE=1.3809, NDCG@10=0.9542
Fold 1, piecewise, ε=1.0: RMSE=1.0472, NDCG@10=0.9580
Fold 1, clipped_laplace, ε=2.0: RMSE=1.2270, NDCG@10=0.9538
Fold 1, bounded_laplace, ε=2.0: RMSE=1.0476, NDCG@10=0.9562
Fold 1, gaussian, ε=2.0: RMSE=1.3574, NDCG@10=0.9537
Fold 1, 

In [ ]:
results_df.to_excel('ldp_results_20pct_1m.xlsx', index=False)
summary.to_excel('ldp_summary_20pct_1m.xlsx', index=False)

In [ ]:
results_df.to_excel('ldp_results_50pct_1m.xlsx', index=False)
summary.to_excel('ldp_summary_50pct_1m.xlsx', index=False)

In [ ]:
# save full results
results_df.to_excel('ldp_results_natural_Jester.xlsx', index=False)

# save summary with mean and std
summary.to_excel('ldp_summary_natural_Jester.xlsx', index=False)